In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import time
import warnings

import joblib
import numpy as np
import polars as pl
import tensorflow as tf
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = 'label'
N_REPEATS = 3
WARMUP_SIZE = 1000


I0000 00:00:1779354455.380945    3645 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779354455.381810    3645 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779354455.429440    3645 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779354456.732278    3645 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para TON-IOT
path_df = '../../DATASETS/dataSets_Reducidos/ton_iot/datos_TON_IoT_redux.csv'

df = pl.read_csv(path_df).drop_nulls()

cols_to_drop = ['label', 'type', 'src_ip', 'dst_ip']

X_df = df.drop(cols_to_drop).to_pandas()
y_np = df[TARGET_COL].to_numpy().astype(np.int8)

indices = np.arange(len(X_df))

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np[train_full_idx],
)

X_train_full_df = X_df.iloc[train_full_idx].copy()
X_test_df = X_df.iloc[test_idx].copy()
X_train_df = X_df.iloc[train_idx].copy()

categorical_cols = ['proto', 'conn_state']
numeric_cols = [col for col in X_df.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)

preprocessor.fit(X_train_df)

X_full_train_np = preprocessor.transform(X_train_full_df).astype(np.float32)
X_test_np = preprocessor.transform(X_test_df).astype(np.float32)
X_full_base = preprocessor.transform(X_df).astype(np.float32)

mlp_scaler = StandardScaler()
mlp_scaler.fit(X_full_train_np)
X_full_scaled_mlp = mlp_scaler.transform(X_full_base).astype(np.float32)

cnn_scaler = MinMaxScaler()
cnn_scaler.fit(X_full_train_np)
X_full_scaled_cnn = cnn_scaler.transform(X_full_base).astype(np.float32)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)

N_MUESTRAS = len(X_full_base)

print(f'Train full: {len(X_full_train_np):,} muestras')
print(f'Test:       {len(X_test_np):,} muestras')
print(f'Total:      {N_MUESTRAS:,} muestras')
print(f'Features:   {X_full_base.shape[1]}')


Train full: 168,834 muestras
Test:       42,209 muestras
Total:      211,043 muestras
Features:   23


In [3]:
# RF

rf_path = '../model/ton-iot/rf_ton_iot.joblib'

rf_model = joblib.load(rf_path)
print('Modelo cargado')

# Warm-up
_ = rf_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))
throughput_rf = N_MUESTRAS / tiempo_total_rf

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s')
print(f'Throughput aproximado: {throughput_rf:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.134, 0.1591, 0.1588]
Tiempo medio de inferencia sobre todo el dataset: 0.1506 s
Throughput aproximado: 1,401,017 muestras/s


In [4]:
# XGBOOST

xgb_path = '../model/ton-iot/xgb_ton_iot.joblib'

xgb_model = joblib.load(xgb_path)
print('Modelo cargado')

# Warm-up
_ = xgb_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))
throughput_xgb = N_MUESTRAS / tiempo_total_xgb

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s')
print(f'Throughput aproximado: {throughput_xgb:,.0f} muestras/s')


Modelo cargado


/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:07:39] WARNING: /__w/xgboost/xgboost/src/gbm/gbtree.cc:402: Changing updater from `grow_gpu_hist` to `grow_quantile_histmaker`.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:07:39] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  setstate(state)
/usr/lib/python3.12/pickle.py:1710: UserWarning: [11:07:39] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  setstate(state)


Tiempos medidos: [0.0643, 0.1423, 0.0364]
Tiempo medio de inferencia sobre todo el dataset: 0.0810 s
Throughput aproximado: 2,605,794 muestras/s


In [5]:
# LIGHTGBM

lgbm_path = '../model/ton-iot/lgbm_ton_iot.joblib'

lgbm_model = joblib.load(lgbm_path)
print('Modelo cargado')

# Warm-up
_ = lgbm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))
throughput_lgbm = N_MUESTRAS / tiempo_total_lgbm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s')
print(f'Throughput aproximado: {throughput_lgbm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.1901, 0.1548, 0.1578]
Tiempo medio de inferencia sobre todo el dataset: 0.1676 s
Throughput aproximado: 1,259,440 muestras/s


In [6]:
# CATBOOST

catboost_path = '../model/ton-iot/catboost_ton_iot.joblib'

catboost_model = joblib.load(catboost_path)
print('Modelo cargado')

# Warm-up
_ = catboost_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))
throughput_catboost = N_MUESTRAS / tiempo_total_catboost

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s')
print(f'Throughput aproximado: {throughput_catboost:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.0912, 0.0917, 0.0908]
Tiempo medio de inferencia sobre todo el dataset: 0.0912 s
Throughput aproximado: 2,313,078 muestras/s


In [7]:
# SVM

svm_path = '../model/ton-iot/svm_ton_iot.joblib'

svm_model = joblib.load(svm_path)
print('Modelo cargado')

# Warm-up
_ = svm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))
throughput_svm = N_MUESTRAS / tiempo_total_svm

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s')
print(f'Throughput aproximado: {throughput_svm:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [0.0505, 0.0375, 0.0358]
Tiempo medio de inferencia sobre todo el dataset: 0.0413 s
Throughput aproximado: 5,113,359 muestras/s


In [8]:
# MLP

mlp_path = '../model/ton-iot/mlp_ton_iot.joblib'

mlp_model = joblib.load(mlp_path)
print('Modelo cargado')

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(WARMUP_SIZE, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))
throughput_mlp = N_MUESTRAS / tiempo_total_mlp

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s')
print(f'Throughput aproximado: {throughput_mlp:,.0f} muestras/s')


E0000 00:00:1779354460.643661    3645 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Modelo cargado


I0000 00:00:1779354461.091650   10361 service.cc:153] XLA service 0x74c4ac0353c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779354461.091668   10361 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1779354461.096768   10361 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779354461.211741   10361 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Tiempos medidos: [0.4191, 0.1564, 0.1802]
Tiempo medio de inferencia sobre todo el dataset: 0.2519 s
Throughput aproximado: 837,747 muestras/s


In [9]:
# CNN

cnn_path = '../model/ton-iot/cnn_ton_iot.joblib'

cnn_model = joblib.load(cnn_path)
print('Modelo cargado')

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(WARMUP_SIZE, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))
throughput_cnn = N_MUESTRAS / tiempo_total_cnn

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s')
print(f'Throughput aproximado: {throughput_cnn:,.0f} muestras/s')


Modelo cargado
Tiempos medidos: [2.3454, 2.0813, 1.9928]
Tiempo medio de inferencia sobre todo el dataset: 2.1399 s
Throughput aproximado: 98,625 muestras/s


In [10]:
# Tabla comparativa final

tabla_comparativa = pd.DataFrame(
    [
        {'Modelo': 'RF', 'Tiempo medio (s)': tiempo_total_rf, 'Muestras/s aprox': throughput_rf},
        {'Modelo': 'XGBoost', 'Tiempo medio (s)': tiempo_total_xgb, 'Muestras/s aprox': throughput_xgb},
        {'Modelo': 'LightGBM', 'Tiempo medio (s)': tiempo_total_lgbm, 'Muestras/s aprox': throughput_lgbm},
        {'Modelo': 'CatBoost', 'Tiempo medio (s)': tiempo_total_catboost, 'Muestras/s aprox': throughput_catboost},
        {'Modelo': 'SVM', 'Tiempo medio (s)': tiempo_total_svm, 'Muestras/s aprox': throughput_svm},
        {'Modelo': 'MLP', 'Tiempo medio (s)': tiempo_total_mlp, 'Muestras/s aprox': throughput_mlp},
        {'Modelo': 'CNN', 'Tiempo medio (s)': tiempo_total_cnn, 'Muestras/s aprox': throughput_cnn},
    ]
).sort_values('Tiempo medio (s)').reset_index(drop=True)

tabla_comparativa['Tiempo medio (s)'] = tabla_comparativa['Tiempo medio (s)'].round(4)
tabla_comparativa['Muestras/s aprox'] = tabla_comparativa['Muestras/s aprox'].round(0).astype(int)

display(tabla_comparativa)


,Modelo,Tiempo medio (s),Muestras/s aprox
0,SVM,0.0413,5113359
1,XGBoost,0.0810,2605794
2,CatBoost,0.0912,2313078
3,RF,0.1506,1401017
4,LightGBM,0.1676,1259440
5,MLP,0.2519,837747
6,CNN,2.1399,98625
